# Fetch MCP server

This notebook deploys the upstream [GoFetch](https://github.com/StacklokLabs/gofetch) MCP server and uses its `fetch` tool to retrieve a web page as Markdown. See the [README](./README.md) for terminal and client configuration instructions.

MCP servers must be enabled on your prokube cluster (v1.9.0 or later).

## Deploy

In [ ]:
!kubectl apply -f fetch-server.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/fetch-example --timeout=3m

## Connect from this Lab

Labs reach the workspace's MCP endpoint through the internal Agent Gateway URL without an API key. This endpoint combines all MCP servers in the workspace and prefixes each tool with its server name, for example `fetch-example_fetch`.

`McpSession` is a minimal MCP client from this repository. Any MCP client library can connect to the same URL.

In [ ]:
%run -n ../../src/pk_helpers/mcp.py

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

session = connect_when_ready(workspace_mcp_url(namespace), tool_prefix="fetch-example_")
session.list_tools()

Fetch a web page:

In [ ]:
result = session.call_tool("fetch-example_fetch", {"url": "https://example.com"})
print(result["content"][0]["text"])

## Connect with an API key

Clients outside the cluster use the server's own route, `https://<your-prokube-domain>/svc/mcp/<workspace>/fetch-example`, with an API key. Create a key with access to `fetch-example` on the **API Keys** page at `https://<your-prokube-domain>/pkui/ai-gateway/keys`.

The next cell prompts for the key unless `MCP_API_KEY` is set and connects to the same route from inside the cluster. Tool names on this route have no prefix.

In [ ]:
%run -n ../../src/pk_helpers/api_key.py

api_key = get_or_create_mcp_api_key()
server_session = connect_when_ready(server_mcp_url(namespace, "fetch-example"), api_key=api_key)
server_session.list_tools()

## Clean up

In [ ]:
!kubectl delete -f fetch-server.yaml